# LLM-proposed reward shaping vs. hand-designed reward -- PPO case study

This notebook is a small, self-contained follow-up experiment inspired by the LLaRA idea
("Efficient Onboard Vision-Language Inference in UAV-Enabled Low-Altitude Economy Networks
via LLM-Enhanced Optimization", arXiv:2510.10028): instead of asking an LLM to act as the
policy directly (which lost badly to PPO in this repo's main comparison), the LLM's job here
is to **propose an improved reward function** for a classical RL agent that still trains and
acts on its own.

**The LLM's proposal** (reasoned about by Claude while pair-programming this repo, not learned
from data): the existing hand-designed reward shapes progress using **Manhattan distance** to
the goal. That's a straight-line heuristic -- it has no idea an obstacle is in the way, so right
next to a wall or a corner it can reward a move that walks the agent into a dead end. A smarter,
still-valid potential function (Ng et al. 1999 potential-based shaping -- doesn't change the
optimal policy either way) is the **true shortest-path distance to the goal through the actual
obstacle layout**, computed with one reverse BFS from the goal per episode (cheap: we already
have a BFS solver in `src/env.py`).

This notebook trains PPO twice -- once with each reward -- and evaluates both on the exact same
test set, so any difference is attributable to the reward design alone (same environment, same
hyperparameters, same training budget).

**No GPU needed.** This notebook only touches `gymnasium` + `stable-baselines3` (classical RL) --
it does NOT load any LLM weights, so it runs fine on CPU, in a few minutes, on Kaggle/Colab/plain
Jupyter alike. (The LLM fine-tuning side of this repo is in `colab_quickstart.ipynb`, which does
need a GPU.)

If you're not opening this notebook from inside an already-cloned repo, edit `REPO_URL` below first.

In [ ]:
REPO_URL = "https://github.com/<your-username>/<your-repo>.git"  # <-- edit this
REPO_DIR = REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')

import os
if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL
%cd $REPO_DIR

In [ ]:
# Lightweight on purpose -- no torch/transformers needed for this notebook.
!pip install -q gymnasium stable-baselines3 pyyaml tqdm

## 1. Generate the dataset (BFS ground truth)

Same train/val/test split the LLM fine-tuning used (`configs/default.yaml`, seed=42) -- skip this
cell if `data/train.jsonl` and `data/test.jsonl` already exist from a previous run.

In [ ]:
!python -m src.data_gen --config configs/default.yaml

## 2. Train PPO with the ORIGINAL hand-designed reward (Manhattan-distance shaping)

~5 minutes on a 2-core CPU. This is the baseline: the reward shaping a human (well, me) wrote by
hand without particularly thinking hard about obstacles.

In [ ]:
!python -m src.train_rl --config configs/default.yaml --timesteps 1200000 \
    --shaping_mode manhattan --output_dir outputs/ppo-pathfinding

## 3. Train PPO with the LLM-PROPOSED reward (BFS-distance shaping)

Same hyperparameters, same training budget, same train instances -- the ONLY thing that changes
is the potential function used for reward shaping (see `src/rl_env.py`'s module docstring for the
exact reasoning).

In [ ]:
!python -m src.train_rl --config configs/default.yaml --timesteps 1200000 \
    --shaping_mode bfs --output_dir outputs/ppo-pathfinding-bfs

## 4. Evaluate both on the exact same test set

Reward shaping only affects *training* -- both agents are scored with the same evaluation harness
(`src/evaluate_rl.py`), so this is an apples-to-apples comparison of the two reward designs.

In [ ]:
!python -m src.evaluate_rl --config configs/default.yaml \
    --model_path outputs/ppo-pathfinding/ppo_model.zip \
    --report_path outputs/eval_report_ppo.json

!python -m src.evaluate_rl --config configs/default.yaml \
    --model_path outputs/ppo-pathfinding-bfs/ppo_model.zip \
    --report_path outputs/eval_report_ppo_bfs.json

In [ ]:
import json

manhattan = json.load(open("outputs/eval_report_ppo.json"))["summary"]
bfs = json.load(open("outputs/eval_report_ppo_bfs.json"))["summary"]

print(f"{'metric':<32}{'manhattan (baseline)':<24}{'bfs (LLM-proposed)':<20}")
for key in ["success_rate", "invalid_move_rate", "unparseable_rate", "avg_length_ratio_on_success"]:
    print(f"{key:<32}{str(manhattan[key]):<24}{str(bfs[key]):<20}")

## Notes

- This is a single run per reward design (one training seed each), not a multi-seed statistical
  study -- treat the gap as a directional signal, not a rigorously significant result. If you want
  more confidence, rerun steps 2-4 a few times with different `--n_envs` seeds and average.
- The reference run (in the accompanying chat/README) found a modest but real improvement from
  the BFS-aware reward: `success_rate` 0.78 -> 0.81, with `invalid_move_rate` and
  `avg_length_ratio_on_success` staying about the same (both already near-optimal with either
  reward on this task). The takeaway isn't "BFS shaping is dramatically better" -- it's that an
  LLM's domain-knowledge critique of a hand-written reward ("this heuristic ignores obstacles")
  can translate into a small, real, and *very cheap* win, without ever putting the LLM in the
  control loop itself.
- See `outputs/comparison_chart.html` in the repo for the full 3-way chart (Base LLM /
  Fine-tuned LoRA / PPO), and the README's "So sánh với RL cổ điển (PPO)" section for the full
  writeup.